# 线性回归的简洁实现

In [ ]:
# 导入必要的库
import torch
from torch import nn
import numpy as np

# 设置随机种子以确保结果可重现
torch.manual_seed(1)

# 打印PyTorch版本
print(torch.__version__)

## 1 生成数据集

In [ ]:
# 定义数据集参数
num_inputs: int = 2  # 特征数量（输入维度）
num_examples: int = 1000  # 样本数量
true_w: list[float] = [2.0, -3.4]  # 真实权重
true_b: float = 4.2  # 真实偏置

# 生成特征数据：从标准正态分布中采样
features = torch.tensor(
    np.random.normal(0, 1, (num_examples, num_inputs)), dtype=torch.float
)

# 根据线性关系生成标签：y = w1*x1 + w2*x2 + b
labels: torch.Tensor = true_w[0] * features[:, 0] + true_w[1] * features[:, 1] + true_b

# 向标签添加噪声，使数据更接近真实情况
labels += torch.tensor(np.random.normal(0, 0.01, size=labels.size()), dtype=torch.float)

## 2 读取数据

In [ ]:
# 导入数据加载相关模块
import torch.utils.data as Data

# 设置批量大小
batch_size: int = 10

# 将特征和标签组合成数据集
dataset = Data.TensorDataset(features, labels)

# 创建数据加载器，用于批量加载数据
data_iter = Data.DataLoader(
    dataset=dataset,  # 要加载的数据集
    batch_size=batch_size,  # 每个批次的大小
    shuffle=True,  # 是否打乱数据（打乱有助于训练）
    num_workers=0,  # 数据加载的线程数（0表示在主进程中加载）
)

In [ ]:
# 从数据加载器中获取一个批次的数据，查看数据结构
X: torch.Tensor
y: torch.Tensor
for X, y in data_iter:
    print(X, "\n", y)  # X是特征，y是对应的标签
    break  # 只查看第一个批次

## 3 定义模型

In [ ]:
# 方法1：通过继承nn.Module自定义网络类
class LinearNet(nn.Module):
    linear: nn.Linear

    def __init__(self, n_feature: int) -> None:
        # 调用父类的构造函数
        super(LinearNet, self).__init__()
        # 定义一个线性层：输入维度为n_feature，输出维度为1
        self.linear = nn.Linear(n_feature, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 定义前向传播：通过线性层
        y: torch.Tensor = self.linear(x)
        return y


# 创建网络实例
net = LinearNet(num_inputs)
# 打印网络结构
print(net)

In [ ]:
# 方法2：使用nn.Sequential构建网络（更简洁的方式）
from collections import OrderedDict

# 写法一：直接传入层
net: nn.Sequential = nn.Sequential(
    nn.Linear(num_inputs, 1)  # 线性回归只需要一个线性层
    # 可以在此添加更多层，如激活函数等
)

# 写法二：使用add_module方法逐层添加
net = nn.Sequential()
net.add_module("linear", nn.Linear(num_inputs, 1))
# 可以继续添加其他层：net.add_module('relu', nn.ReLU())

# 写法三：使用OrderedDict按顺序添加层（可以指定层名）
net = nn.Sequential(
    OrderedDict(
        [
            ("linear", nn.Linear(num_inputs, 1))  # 添加线性层并命名为'linear'
            # 可以添加更多层：('relu', nn.ReLU())
        ]
    )
)

# 打印整个网络结构和第一个层的信息
print(net)
print(net[0])

In [ ]:
# 查看网络中的可学习参数
# 这些参数是随机初始化的（默认初始化）
param: torch.nn.Parameter
for param in net.parameters():
    print(param)  # 第一个是权重参数，第二个是偏置参数

## 4 初始化模型参数

In [ ]:
# 导入初始化模块
from torch.nn import init

# 先拿到线性层并做类型判断，避免类型检查报错
layer: nn.Module = net[0]
if isinstance(layer, nn.Linear):
    # 使用正态分布初始化权重：均值为0，标准差为0.01
    init.normal_(layer.weight, mean=0.0, std=0.01)
    # 将偏置初始化为0
    init.constant_(layer.bias, val=0.0)
    # 另一种初始化偏置的方法：layer.bias.data.fill_(0)

In [ ]:
# 再次查看初始化后的参数
for param in net.parameters():
    print(param)  # 权重现在接近0，偏置为0

## 5 定义损失函数

In [ ]:
# 定义均方误差（MSE）损失函数
# 适用于回归问题，计算预测值和真实值之间的均方误差
loss: nn.MSELoss = nn.MSELoss()

## 6 定义优化算法

In [ ]:
# 导入优化器模块
import torch.optim as optim

# 使用随机梯度下降（SGD）优化器
# 参数：网络的所有可学习参数，学习率设为0.03
optimizer: optim.SGD = optim.SGD(net.parameters(), lr=0.03)

# 打印优化器的配置信息
print(optimizer)

In [ ]:
# 高级用法：为网络的不同部分设置不同的学习率
# optimizer = optim.SGD([
#     # 如果不指定学习率，则使用最外层的默认学习率
#     {'params': net.subnet1.parameters()},  # 使用默认学习率0.03
#     {'params': net.subnet2.parameters(), 'lr': 0.01}  # 指定学习率为0.01
# ], lr=0.03)  # 默认学习率

In [ ]:
# 调整学习率的示例（动态学习率调整）
# for param_group in optimizer.param_groups:
#     param_group['lr'] *= 0.1  # 将学习率降低为原来的0.1倍

## 7 训练模型

In [ ]:
# 设置训练轮数
num_epochs: int = 3

# 开始训练循环
for epoch in range(1, num_epochs + 1):
    # 遍历数据加载器中的每个批次
    loss_value: torch.Tensor = torch.tensor(float("nan"))
    for X, y in data_iter:
        # 前向传播：通过网络计算预测值
        output: torch.Tensor = net(X)

        # 计算损失：比较预测值和真实值
        # y.view(-1, 1)将标签reshape为与output相同的形状
        loss_value = loss(output, y.view(-1, 1))

        # 梯度清零：清除上一批次的梯度，防止梯度累积
        optimizer.zero_grad()  # 等价于net.zero_grad()

        # 反向传播：计算梯度
        loss_value.backward()

        # 更新参数：根据梯度更新网络参数
        optimizer.step()

    # 每轮训练后打印损失值
    print("epoch %d, loss: %f" % (epoch, loss_value.item()))

In [ ]:
# 训练完成后，查看学到的参数

# 获取网络的线性层（因为是Sequential的第一个层）
dense: nn.Module = net[0]
assert isinstance(dense, nn.Linear)

# 比较真实权重和学到的权重
print("真实权重:", true_w, "\n学到的权重:", dense.weight.data)

# 比较真实偏置和学到的偏置
print("真实偏置:", true_b, "\n学到的偏置:", dense.bias.data)

In [ ]:
# 代码结束